# The Scenario
You are building an app that suggests how much Steak and Beans a user should eat.

1. The Neural Network (MSE): Predicts what the user wants to eat based on their taste profile (e.g., the user loves Steak).

2. The LP Objective (Cost): We want the suggested meal to be cheap.

-   Steak costs $10.00 per unit.

-   Beans cost $1.00 per unit.

3. The Constraints (Health): The meal must have at least 20g of Protein.

-   Steak provides 10g protein/unit.

-   Beans provide 5g protein/unit.

## The Conflict
- User/MSE wants: Lots of Steak (Tasty).
- LP Objective wants: Only Beans (Cheap).
- Constraint wants: Enough food to hit 20g protein.

In [212]:
import torch
import torch.nn as nn
import torch.optim as optim

In [213]:
# --- 1. Setup Constants ---
# Costs (Objective Vector c)
# [Steak, Beans] -> Steak is $10, Beans is $1
cost_vector = torch.tensor([10.0, 1.0]) 

# Protein Content (Constraint Matrix A)
# [Steak, Beans] -> Steak=10g, Beans=5g
protein_content = torch.tensor([10.0, 5.0]) 

# Minimum Protein Required (b)
min_protein = 20.0 

# User Preference (Target for MSE)
# The user LOVES Steak. They ideally want 3 units of Steak and 0 Beans.
user_preference = torch.tensor([[3.0, 0.0]])

In [214]:
# --- 2. The Model ---
# A simple model that takes a "User ID" (dummy input) and predicts Food Amounts
model = nn.Sequential(
    nn.Linear(10, 10),
    nn.ReLU(),
    nn.Linear(10, 2),
    nn.ReLU() # Important: We can't eat negative food!
)
model

Sequential(
  (0): Linear(in_features=10, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=2, bias=True)
  (3): ReLU()
)

In [215]:
optimizer = optim.Adam(model.parameters(), lr=0.02)

In [216]:
# --- 3. The "Three-Head" Loss Function ---
def dietitian_loss(y_pred, y_target):
    # A. MSE Loss (Satisfaction)
    # Penalize deviating from what the user likes (3 Steak, 0 Beans)
    loss_satisfaction = nn.MSELoss()(y_pred, y_target)
    
    # B. LP Objective (Cost Minimization)
    # Calculate total bill: (Amount * Price)
    # We want this to be small.
    total_cost = torch.matmul(y_pred, cost_vector)
    loss_cost = torch.mean(total_cost)
    
    # C. LP Constraint (Health)
    # We need: Protein_Amount >= 20
    # Violation if: 20 - Protein_Amount > 0
    current_protein = torch.matmul(y_pred, protein_content)
    violation = torch.relu(min_protein - current_protein)
    loss_health = torch.mean(violation**2)
    
    # --- Weighted Sum ---
    # We weight "Health" highest because it's a hard rule.
    # We balance "Satisfaction" vs "Cost".
    return (10 * loss_satisfaction) + (15 * loss_cost) + (5.0 * loss_health)

In [217]:
# --- 4. Training ---
dummy_input = torch.randn(1, 10)

print(f"{'Epoch':<5} | {'Steak':<8} | {'Beans':<8} | {'Protein':<8} | {'Cost ($)':<8}")
print("-" * 55)

for epoch in range(1000):
    optimizer.zero_grad()
    y_pred = model(dummy_input)
    
    loss = dietitian_loss(y_pred, user_preference)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        d = y_pred[0].detach().numpy()
        prot = (d[0]*10 + d[1]*5)
        cost = (d[0]*10 + d[1]*1)
        print(f"{epoch:<5} | {d[0]:.2f}     | {d[1]:.2f}     | {prot:.2f}     | ${cost:.2f}")

# --- Final Analysis ---
print("-" * 55)
final = model(dummy_input)[0].detach()
print("FINAL RECOMMENDATION:")
print(f"Steak: {final[0]:.2f} units")
print(f"Beans: {final[1]:.2f} units")

Epoch | Steak    | Beans    | Protein  | Cost ($)
-------------------------------------------------------
0     | 0.21     | 0.00     | 2.10     | $2.10
20    | 2.85     | 0.00     | 28.49     | $28.49
40    | 1.89     | 0.00     | 18.91     | $18.91
60    | 1.89     | 0.00     | 18.92     | $18.92
80    | 1.89     | 0.00     | 18.88     | $18.88
100   | 1.86     | 0.00     | 18.65     | $18.65
120   | 1.86     | 0.00     | 18.58     | $18.58
140   | 1.86     | 0.00     | 18.62     | $18.62
160   | 1.86     | 0.00     | 18.61     | $18.61
180   | 1.86     | 0.00     | 18.61     | $18.61
200   | 1.86     | 0.00     | 18.61     | $18.61
220   | 1.86     | 0.00     | 18.61     | $18.61
240   | 1.86     | 0.00     | 18.61     | $18.61
260   | 1.86     | 0.00     | 18.61     | $18.61
280   | 1.86     | 0.00     | 18.61     | $18.61
300   | 1.86     | 0.00     | 18.61     | $18.61
320   | 1.86     | 0.00     | 18.61     | $18.61
340   | 1.86     | 0.00     | 18.61     | $18.61
360   | 1.86  